In [2]:
from numpy import argmax

In [3]:
def compute_s(Htr_PosOnes, e_pos, r, t, v):
    '''
    Compute syndrome exploiting sparsity 
    '''
    s = vector(GF(2),r)
    
    for i in range(t):
        pos = e_pos[i]
        
        block = pos//r
        shift = pos%r
        
        for j in range(v):
            toggle_pos = (Htr_PosOnes[block][j]+shift)%r
            s[toggle_pos] += 1
    
    return s

def compare_errors(e, e_pos):
    
    for pos in e_pos:
        e[pos] += 1
    
    num_errors = e.list().count(1)
    
    if num_errors == 0:
        return 0
    else:
        return 1

#### Counters computation functions

In [4]:
def counter_sparse_to_dense(Htr_PosOnes, H_PosOnes, s, r, v):
    '''
    - Htr_PosOnes = posizioni delle colonne di H
    - H_PosOnes = posizioni delle righe
    - s: sindrome (densa, definita su interi)
    '''
        
    sigma = vector(ZZ,2*r) #we later accumulate here
    
    for i in range(2):
        for j in range(v):
            
            shift_val = H_PosOnes[i][j] #shift value
            
            #shift syndrome
            delta_sigma = vector(ZZ,r) #delta for counters    
            for ell in range(r): #shifto la cazzo di sindrome
                new_pos = (ell+shift_val)%r;
                delta_sigma[new_pos] = s[ell]
            
            #accumula
            sigma[i*r:(i+1)*r] += delta_sigma
    
    return sigma

##################################

def counters_update_slow(Htr_PosOnes, H_PosOnes, s, r, v, pos, sigma):
    
    '''
    This is the update function for BFmax, using only sparse operations
    pos = position to flip
    '''
    block = pos//r #block where error lies
    shift = pos%r
    
    for i in range(v):
        row_index = (Htr_PosOnes[block][i] + shift)%r
        d = s[row_index] #if d = 1, decrease counters; otherwise, increase

        if d == 1:
            upc_increment = -1
        else:
            upc_increment = 1
        
        #update all counters
        for j in range(2):
            for ell in range(v):
                pos_toggle = j*r + (H_PosOnes[j][ell] + row_index)%r
                sigma[pos_toggle] += upc_increment
    
    return sigma

##################################

def fixed_counters_decrement(Htr_PosOnes, H_PosOnes, r, v):
    '''
    Decrements for each bit flip, up to a cyclic shift
    '''
    fixed_decr = []
    for block in range(2):
        #Set up syndrome for current block
        dense_s_block = vector(ZZ,r)
        for i in range(v):
            pos_toggle = Htr_PosOnes[block][i]
            dense_s_block[pos_toggle] = 1
        # alla fine di questo abbiamo densificato le colonne
        # ottengo il decremento accumulando lo shift di HTr[0] per le H_PosOnes
        decr = counter_sparse_to_dense(Htr_PosOnes, H_PosOnes, dense_s_block, r, v)
        fixed_decr.append(decr)
    
    return fixed_decr

###############################

def counters_update_steroids(Htr_PosOnes, H_PosOnes, fixed_decr, s, r, v, pos, sigma):
    '''
    This is the update function for BFmax, using the best idea we cam up with
    fixed_decr 
    '''
    block = pos//r #block where error lies
    shift = pos%r
    
    # Do update due to decrements
    # Shift of the fixed contribution vector equal to the flipped position
    shifted_incr = vector(ZZ, 2*r)
    for i in range(2):
        for j in range(r):
            new_pos = i*r + ((j+shift)%r)
            shifted_incr[new_pos] = fixed_decr[block][i*r+j]
    
    sigma += (-shifted_incr)
    
    #Now, correct 
    for i in range(v):
        row_index = (Htr_PosOnes[block][i] + shift)%r
        d = s[row_index] #if d = 1, decrease counters; otherwise, increase
        
        if d == 0:

            #update all counters
            for j in range(2):
                for ell in range(v):
                    pos_toggle = j*r + (H_PosOnes[j][ell] + row_index)%r
                    sigma[pos_toggle] += 2
    
    return sigma

In [5]:
def bfmax_bit_slice(Htr_PosOnes, H_PosOnes, s, r, v, num_iter_max):
    '''
    Every iteration, we recompute counters from scratch
    Here, s must 
    '''
    
    dec_e = vector(GF(2), 2*r) #error estimate
    
    w_s = s.list().count(1) #syndrome weight
    num_iter = 0 #num of performed iterations
    
    while (w_s != 0) & (num_iter < num_iter_max):
        
        #Compute counters
        sigma = counter_sparse_to_dense(Htr_PosOnes, H_PosOnes, s.change_ring(ZZ), r, v)

        #Find position to flip
        pos = argmax(sigma)
        
        #Flip position in dec_e
        dec_e[pos] += 1
        
        #Flip syndrome
        block = pos // r
        shift = pos % r
        for i in range(v):
            pos_toggle = (Htr_PosOnes[block][i]+shift)%r
            s[pos_toggle] += 1
        
        #Update syndrome weight and number of iterations
        w_s = s.list().count(1) #syndrome weight        
        num_iter += 1
        
    return dec_e

########################################################

def bfmax_slow_update(Htr_PosOnes, H_PosOnes, s, r, v, num_iter_max):
    '''
    Every iteration, we recompute counters as we did in our paper
    '''
    
    dec_e = vector(GF(2), 2*r) #error estimate
    
    w_s = s.list().count(1) #syndrome weight
    num_iter = 0 #num of performed iterations
    
    #Compute counters, for the first time, from scratch
    sigma = counter_sparse_to_dense(Htr_PosOnes, H_PosOnes, s.change_ring(ZZ), r, v)
        
    while (w_s != 0) & (num_iter < num_iter_max):
        
        #Find position to flip
        pos = argmax(sigma)
        
        #Flip position in dec_e
        dec_e[pos] += 1
        
        #Update counters
        sigma = counters_update_slow(Htr_PosOnes, H_PosOnes, s, r, v, pos, sigma);

        #Flip syndrome
        block = pos // r
        shift = pos % r
        for i in range(v):
            pos_toggle = (Htr_PosOnes[block][i]+shift)%r
            s[pos_toggle] += 1
        
        #Update syndrome weight and number of iterations
        w_s = s.list().count(1) #syndrome weight        
        num_iter += 1
        
    return dec_e

########################################################

def bfmax_steroids(Htr_PosOnes, H_PosOnes, fixed_updates, s, r, v, num_iter_max):
    '''
    We test the new idea about updating counters
    '''
    
    dec_e = vector(GF(2), 2*r) #error estimate
    
    w_s = s.list().count(1) #syndrome weight
    num_iter = 0 #num of performed iterations
    
    #Compute counters, for the first time, from scratch
    sigma = counter_sparse_to_dense(Htr_PosOnes, H_PosOnes, s.change_ring(ZZ), r, v)
    
    while (w_s != 0) & (num_iter < num_iter_max):
        
        #Find position to flip
        pos = argmax(sigma)

        print(f"At iteration {num_iter} the value of the max counter is {sigma[pos]}")

        
        #Flip position in dec_e
        dec_e[pos] += 1
        
        #Update counters
        sigma = counters_update_steroids(Htr_PosOnes, H_PosOnes, fixed_updates, s, r, v, pos, sigma);

        #Flip syndrome
        block = pos // r
        shift = pos % r
        for i in range(v):
            pos_toggle = (Htr_PosOnes[block][i]+shift)%r
            s[pos_toggle] += 1
        
        #Update syndrome weight and number of iterations
        w_s = s.list().count(1) #syndrome weight        
        num_iter += 1
        
    return dec_e

# Do some sanity checks

In [6]:
r = 50; v = 5; t = 3;

#Sampling colonne
Htr_PosOnes = [Combinations(r,v).random_element(),Combinations(r,v).random_element()] 

#Calcolo righe
H_PosOnes = []
for i in range(2):
    positions = [((r-a)%r) for a in Htr_PosOnes[i]] #fai il trasposto e riduci modulo r
    H_PosOnes.append(positions)

## Prova

In [7]:
e_pos = Combinations(r*2, t).random_element()
s = compute_s(Htr_PosOnes, e_pos, r, t, v)
    
sigma = counter_sparse_to_dense(Htr_PosOnes, H_PosOnes, s.change_ring(ZZ), r, v)

Sanity check: densifico tutto quanto, faccio i contatori facendo letteralmente $\sigma * H$. It's very slow, do it only with very small parameters.

In [8]:
#Verify syndrome
H_dense = matrix(GF(2), r, 2*r)
for row_index in range(r):
    for i in range(2):
        for j in range(v):
            pos = (H_PosOnes[i][j]+row_index)%r
            H_dense[row_index,i*r + pos] = 1

e_dense = matrix(GF(2),2*r)
for i in e_pos:
    e_dense[0,i] = 1;
    
s_dense = e_dense * H_dense.transpose()

check_syndrome = 0
for i in range(r):
    if s_dense[0,i] != s[i]:
        check_syndrome += 1
if check_syndrome != 0:
    print("Syndrome is wrong")
else:
    print("Syndrome is ok")

#Verify counters
check_counters = 0
for i in range(2*r):
    sigma_i = s_dense.change_ring(ZZ)*H_dense[:,i]
    if sigma_i[0,0] != sigma[i]:
        check_counters += 1
        
if check_counters != 0:
    print("Counters are wrong")
else:
    print("Counters are ok")

Syndrome is ok
Counters are ok


# Test decoding

We compare the three different implementations and see if they work in the same way.
Notice that when decoding fails we should get different values, but we're getting the very same values because the search for the maximum coutner is currently deterministic

In [11]:
r = 12323; v = 71;

#Sampling colonne
Htr_PosOnes = [Combinations(r,v).random_element(),Combinations(r,v).random_element()] 

#Calcolo righe
H_PosOnes = []
for i in range(2):
    positions = [((r-a)%r) for a in Htr_PosOnes[i]] #fai il trasposto e riduci modulo r
    H_PosOnes.append(positions)
    
#Fixed counters decrements
block_decrements = fixed_counters_decrement(Htr_PosOnes, H_PosOnes, r, v)
print(block_decrements)
print(argmax(block_decrements))

[(71, 0, 1, 1, 0, 2, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 2, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 2, 0, 1, 1, 0, 0, 2, 1, 0, 2, 0, 0, 1, 1, 1, 0, 0, 1, 0, 2, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 2, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 2, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 2, 0, 0, 1, 0, 0, 0, 1, 2, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0

We test several error weights and all decoders

In [30]:
import time

num_decodes = 10

for t in [134]:
    
    #DFR values
    dfr_t_1 = 0; dfr_t_2 = 0    ; dfr_t_3 = 0
    
    #Timings
    T_1 = 0; T_2 = 0; T_3 = 0
    
    for idx in range(num_decodes):
        
        #Sample error positions
        e_pos = Combinations(2*r,t).random_element()
        
        #Compute syndrome (and do copies because we test several decoders)
        s_1 = compute_s(Htr_PosOnes, e_pos, r, t, v)
        s_2 = copy(s_1)
        s_3 = copy(s_1)
        
        #Decode (implementation 1)
        start_1 = time.time()
        #dec_e_1 = bfmax_bit_slice(Htr_PosOnes, H_PosOnes, s_1, r, v, 2*t)
        end_1 = time.time()
        T_1 += (end_1 - start_1)
        
        #Decode (implementation 2)
        start_2 = time.time()
        #dec_e_2 = bfmax_slow_update(Htr_PosOnes, H_PosOnes, s_2, r, v, 2*t)
        end_2 = time.time()
        T_2 += (end_2 - start_2)
        
        #Decode (implementation 3)
        start_3 = time.time()
        dec_e_3 = bfmax_steroids(Htr_PosOnes, H_PosOnes, block_decrements, s_3, r, v, 2*t)
        end_3 = time.time()
        T_3 += (end_3 - start_3)
        
        if (dec_e_1 != dec_e_2) | (dec_e_3 != dec_e_1):
            print("---> Decoded errors are different")
            
        #Compare with true error vector
        ok_1 = compare_errors(dec_e_1, e_pos)
        ok_2 = compare_errors(dec_e_2, e_pos)
        ok_3 = compare_errors(dec_e_3, e_pos)
            
        if ok_1:
            dfr_t_1 += 1
        if ok_2:
            dfr_t_2 += 1
        if ok_3:
            dfr_t_3 += 1

    print("t = ",t,", DFR: ",N(dfr_t_1/num_decodes),", D",N(dfr_t_2/num_decodes),", ",N(dfr_t_3/num_decodes))
    print("Timings:", T_1/num_decodes, T_2/num_decodes, T_3/num_decodes)
    print("---------------")

At iteration 0 the value of the max counter is 54
At iteration 1 the value of the max counter is 54
At iteration 2 the value of the max counter is 53
At iteration 3 the value of the max counter is 52
At iteration 4 the value of the max counter is 50
At iteration 5 the value of the max counter is 50
At iteration 6 the value of the max counter is 50
At iteration 7 the value of the max counter is 50
At iteration 8 the value of the max counter is 49
At iteration 9 the value of the max counter is 49
At iteration 10 the value of the max counter is 50
At iteration 11 the value of the max counter is 49
At iteration 12 the value of the max counter is 50
At iteration 13 the value of the max counter is 48
At iteration 14 the value of the max counter is 49
At iteration 15 the value of the max counter is 48
At iteration 16 the value of the max counter is 48
At iteration 17 the value of the max counter is 47
At iteration 18 the value of the max counter is 49
At iteration 19 the value of the max coun

KeyboardInterrupt: 